# Notebook 17: Deep Learning Hyperparameter Experiments



## 1. Dataset and split
The generated binary dataset has nontrivial feature structure. Standardization is fit only on training features to avoid leaking validation/test statistics. The final test split is held aside until model selection is complete.

In [1]:
import random, numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

SEED = 23
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X, y = make_classification(n_samples=2400, n_features=20, n_informative=12, n_redundant=4, class_sep=1.0, random_state=SEED)
X_train_raw, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)
scaler = StandardScaler().fit(X_train_raw)
X_train = torch.tensor(scaler.transform(X_train_raw), dtype=torch.float32); y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_val = torch.tensor(scaler.transform(X_val_raw), dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)
X_test = torch.tensor(scaler.transform(X_test_raw), dtype=torch.float32)
print(f'Device={device}; train={len(y_train)}, validation={len(y_val)}, test={len(y_test)}')

Device=cpu; train=1680, validation=360, test=360


## 2. Define configurations and training procedure
The output layer returns logits, so `BCEWithLogitsLoss` is numerically stable and no sigmoid is needed during training. We log validation loss and accuracy per experiment. A fixed seed makes comparisons more repeatable, though mini-batch ordering and hardware may still introduce small variation.

In [2]:
experiments = [
    {'name':'A: shallow SGD', 'lr':0.01, 'batch':64, 'epochs':12, 'widths':[32], 'dropout':0.0, 'optimizer':'SGD', 'activation':'ReLU', 'weight_decay':0.0},
    {'name':'B: deeper Adam', 'lr':0.003, 'batch':64, 'epochs':12, 'widths':[64, 32], 'dropout':0.0, 'optimizer':'Adam', 'activation':'ReLU', 'weight_decay':1e-4},
    {'name':'C: dropout Adam', 'lr':0.003, 'batch':32, 'epochs':15, 'widths':[64, 32], 'dropout':0.30, 'optimizer':'Adam', 'activation':'ReLU', 'weight_decay':1e-4},
    {'name':'D: wide AdamW', 'lr':0.001, 'batch':128, 'epochs':15, 'widths':[128, 64], 'dropout':0.20, 'optimizer':'AdamW', 'activation':'GELU', 'weight_decay':1e-3},
    {'name':'E: higher LR Adam', 'lr':0.03, 'batch':64, 'epochs':12, 'widths':[64, 32], 'dropout':0.10, 'optimizer':'Adam', 'activation':'Tanh', 'weight_decay':1e-4}
]

In [3]:
def build_model(config):
    layers = []; in_features = X_train.shape[1]
    activation_cls = {'ReLU': nn.ReLU, 'Tanh': nn.Tanh, 'GELU': nn.GELU}[config['activation']]
    for width in config['widths']:
        layers.extend([nn.Linear(in_features, width), activation_cls()])
        if config['dropout'] > 0: layers.append(nn.Dropout(config['dropout']))
        in_features = width
    layers.append(nn.Linear(in_features, 1))
    return nn.Sequential(*layers).to(device)

def run_experiment(config):
    model = build_model(config); criterion = nn.BCEWithLogitsLoss()
    train_loader = DataLoader(TensorDataset(X_train, y_train_t), batch_size=config['batch'], shuffle=True)
    opt_cls = {'SGD': torch.optim.SGD, 'Adam': torch.optim.Adam, 'AdamW': torch.optim.AdamW}[config['optimizer']]
    optimizer = opt_cls(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    for _ in range(config['epochs']):
        model.train()
        for features, targets in train_loader:
            features, targets = features.to(device), targets.to(device)
            optimizer.zero_grad(); loss = criterion(model(features), targets); loss.backward(); optimizer.step()
    model.eval()
    with torch.no_grad():
        logits = model(X_val.to(device)); val_loss = criterion(logits, y_val_t.to(device)).item()
        val_accuracy = ((logits >= 0).cpu().numpy().reshape(-1) == y_val).mean()
    return model, val_loss, val_accuracy

runs = []
for config in experiments:
    model, val_loss, val_acc = run_experiment(config)
    runs.append({'config':config, 'model':model, 'val_loss':val_loss, 'val_accuracy':val_acc})
    print(f"{config['name']}: validation accuracy={val_acc:.3f}, loss={val_loss:.4f}")

A: shallow SGD: validation accuracy=0.769, loss=0.5863
B: deeper Adam: validation accuracy=0.956, loss=0.1324
C: dropout Adam: validation accuracy=0.956, loss=0.1275
D: wide AdamW: validation accuracy=0.936, loss=0.1831
E: higher LR Adam: validation accuracy=0.947, loss=0.1827


## 3. Experiment table and configuration selection
The requested columns are shown alongside epochs, activation, dropout, and weight decay so the comparison is interpretable. Selection uses validation accuracy, with validation loss as a tie-breaker; if two runs are effectively tied, prefer the simpler/faster configuration.

In [4]:
rows = []
for run in runs:
    c = run['config']
    rows.append({'Experiment':c['name'], 'Learning Rate':c['lr'], 'Batch Size':c['batch'], 'Architecture':str(c['widths']),
                 'Optimizer':c['optimizer'], 'Accuracy':run['val_accuracy'], 'Loss':run['val_loss'], 'Epochs':c['epochs'],
                 'Activation':c['activation'], 'Dropout':c['dropout'], 'Weight Decay':c['weight_decay']})
results = pd.DataFrame(rows).sort_values(['Accuracy','Loss'], ascending=[False, True]).reset_index(drop=True)
display(results)
best_run = max(runs, key=lambda r: (r['val_accuracy'], -r['val_loss']))
print('Selected on validation:', best_run['config']['name'])
print('Selection rationale: highest validation accuracy; validation loss breaks close ties. Check whether the gain justifies additional complexity.')

,Experiment,Learning Rate,Batch Size,Architecture,Optimizer,Accuracy,Loss,Epochs,Activation,Dropout,Weight Decay
0,C: dropout Adam,0.003,32,"[64, 32]",Adam,0.955556,0.127541,15,ReLU,0.3,0.0001
1,B: deeper Adam,0.003,64,"[64, 32]",Adam,0.955556,0.132362,12,ReLU,0.0,0.0001
2,E: higher LR Adam,0.030,64,"[64, 32]",Adam,0.947222,0.182721,12,Tanh,0.1,0.0001
3,D: wide AdamW,0.001,128,"[128, 64]",AdamW,0.936111,0.183112,15,GELU,0.2,0.0010
4,A: shallow SGD,0.010,64,[32],SGD,0.769444,0.586284,12,ReLU,0.0,0.0000


Selected on validation: C: dropout Adam
Selection rationale: highest validation accuracy; validation loss breaks close ties. Check whether the gain justifies additional complexity.


## 4. One-time test evaluation and interpretation
A validation winner is not guaranteed to be best on every sample. We now evaluate the selected configuration once on the untouched test split.

Discuss the table: a higher learning rate may converge quickly but become unstable; larger batches smooth gradients but produce fewer updates per epoch; deeper/wider networks add capacity; dropout and weight decay constrain capacity; optimizer and activation change optimization dynamics. These effects depend on data and should be judged from measured curves/scores rather than assumed.

In [5]:
selected_model = best_run['model'].eval()
with torch.no_grad():
    test_logits = selected_model(X_test.to(device)).cpu().numpy().reshape(-1)
    test_predictions = (test_logits >= 0).astype(int)
test_accuracy = accuracy_score(y_test, test_predictions)
test_loss = nn.BCEWithLogitsLoss()(torch.tensor(test_logits), torch.tensor(y_test, dtype=torch.float32)).item()
print(f"Selected config: {best_run['config']['name']}")
print(f'Test accuracy={test_accuracy:.4f}; test loss={test_loss:.4f}')
print('Report whether test performance is close to validation. A large drop may indicate validation over-selection or sampling variation.')

Selected config: C: dropout Adam
Test accuracy=0.9528; test loss=0.1287
Report whether test performance is close to validation. A large drop may indicate validation over-selection or sampling variation.
